In [1]:
# set up
import pandas as pd
from openai import OpenAI
import os
from dotenv import load_dotenv
import time
import json

#revise work directory
os.chdir("/Users/xwei/Desktop/AI-assisted-review-")
project_root = os.getcwd()

dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path=dotenv_path, override=True)
# Initialize the client with the API key
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

#start = time.perf_counter()
#end = time.perf_counter()
#elapsed = end - start
#print(f"Code took {elapsed:.4f} seconds")

In [8]:
#data for phase 1
#df = pd.read_excel("Phase1.xlsx") #replace this with Phase1.xlsx

#for phase 2, "The Phase_2_combined dataset currently includes the is.included column, which is the human validation column. Please remove this column first and save the dataset as a new version for AI testing. This will help prevent data leakage in the LLMs’ decision-making." 
#df = pd.read_csv("Phase_2_combined.csv") 
#df = df.drop(columns=["is.included"])
#df.to_csv("Phase_2_combined_no_label.csv", index=False)

df = pd.read_csv("Phase_2_combined_no_label.csv") 
df

,Title,Authors,Abstract,Published Year,Published Month,Journal,Volume,Issue,Pages,Accession Number,DOI,Ref,Covidence #,Study,Notes,Tags
0,Nonvocational Outcomes from a Randomized Contr...,"Ferguson, Kristin M.",Purpose: This randomized controlled trial comp...,2018.0,NaN,Research on Social Work Practice,28,5,603-618,"SAGE Publications. 2455 Teller Road, Thousand ...",10.1177/1049731517709076,NaN,#5,Ferguson 2018,NaN,NaN
1,Increasing Identification of Homeless Students...,"Shephard, Daniel D.; Hall, Crystal C.; Lambert...",Over 1.5 million students in the United States...,2021.0,NaN,Educational Researcher,50,4,239-248,"SAGE Publications. 2455 Teller Road, Thousand ...",10.3102/0013189X20981067,NaN,#21,Shephard 2021,NaN,NaN
2,'I Need to Get My Culture Back': Youth and Pro...,Charlene Kuo; Michelle Jasczynski; Jee Hun Yoo...,There is growing interest in decolonizing sexu...,2023.0,NaN,Prevention Science,24,2,209-221,Springer. Available from: Springer Nature. One...,10.1007/s11121-023-01573-7,NaN,#40,CharleneKuo 2023,NaN,NaN
3,Extended foster care and homelessness: Assessi...,"Spindle-Jackson, A.; Byrne, T.; Collins, M.E.",Youth aging out of foster care experience high...,2024.0,NaN,Child. Youth Serv. Rev.,164,NaN,NaN,NaN,10.1016/j.childyouth.2024.107820,NaN,#77,Spindle-Jackson 2024,NaN,NaN
4,A feasibility (pilot) mixed methods study of a...,"Vasudev, A.; Ionson, E.; Sathiaselan, J.; That...",Background: Various service provision models f...,2024.0,NaN,Pilot Feasibility Stud.,10,1,NaN,NaN,10.1186/s40814-024-01452-0,NaN,#79,Vasudev 2024,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1837,An eight-year epidemiologic study of head and ...,"Qian, Xu; Nguyen, Duc T.; Albers, Andreas E.; ...",BACKGROUND: Head and neck tuberculosis (HNTB) ...,2019.0,NaN,Tuberculosis (Edinb),116S,NaN,S71-S77,NaN,10.1016/j.tube.2019.04.013,NaN,#3968,Qian 2019,NaN,NaN
1838,Assessing Racial Disparities in HCV Infection ...,"McGonigle, Keanan; Carley, Tess; Hoff, Clarissa",OBJECTIVES: This study assessed racial dispari...,2018.0,NaN,J Racial Ethn Health Disparities,5,5,1052-1058,NaN,10.1007/s40615-017-0453-y,NaN,#3969,McGonigle 2018,NaN,NaN
1839,Healthcare resource utilization and costs asso...,"Samant, Salome; Chen, Edith; Carias, Cristina;...",AIM: To investigate hepatitis A-related health...,2024.0,NaN,J Med Econ,27,1,1046-1052,NaN,10.1080/13696998.2024.2384263,NaN,#3970,Samant 2024,NaN,NaN
1840,Technology Access and Perceptions of Telehealt...,"Ertl, Melissa M.; Jones, Alexis; Hickson, Robe...",PURPOSE: This study examined access to technol...,2024.0,NaN,J Adolesc Health,74,3,582-590,NaN,10.1016/j.jadohealth.2023.09.019,NaN,#3971,Ertl 2024,NaN,NaN


In [7]:
# Retrieve the list of models
models_response = client.models.list()

# Print each model's details line by line
for model in models_response.data:
    print(f"Model ID: {model.id}")
    print(f"Created: {model.created}")
    print(f"Object: {model.object}")
    print(f"Owned by: {model.owned_by}")
    print("-" * 40)


Model ID: gpt-4-0613
Created: 1686588896
Object: model
Owned by: openai
----------------------------------------
Model ID: gpt-4
Created: 1687882411
Object: model
Owned by: openai
----------------------------------------
Model ID: gpt-3.5-turbo
Created: 1677610602
Object: model
Owned by: openai
----------------------------------------
Model ID: gpt-5.4-mini
Created: 1773451123
Object: model
Owned by: system
----------------------------------------
Model ID: gpt-5.4
Created: 1772691852
Object: model
Owned by: system
----------------------------------------
Model ID: gpt-5.4-nano-2026-03-17
Created: 1773450837
Object: model
Owned by: system
----------------------------------------
Model ID: gpt-5.4-nano
Created: 1773450870
Object: model
Owned by: system
----------------------------------------
Model ID: gpt-5.4-mini-2026-03-17
Created: 1773451076
Object: model
Owned by: system
----------------------------------------
Model ID: davinci-002
Created: 1692634301
Object: model
Owned by: syste

In [ ]:
# some column names difference between phase 1 and phase 2 datasets, so we will use the following column names in the code. Please make sure to update the column names in the dataset accordingly before running the code.
# For phase 1:
# title
# abstract
# publication_year
# journal_name
# author
#         f"Author Affiliation: {row['author_affiliation']}\n"
#         f"Keywords: {row['keywords']}"

# For phase 2:
# Title
# Abstract
# Published Year
# Journal
# Authors
# * no author affiliation and keywords

df.rename(columns={"Title": "title", "Abstract": "abstract", "Published Year": "publication_year", "Journal": "journal_name", "Authors": "author"}, inplace=True)
df

,title,author,abstract,publication_year,Published Month,journal_name,Volume,Issue,Pages,Accession Number,DOI,Ref,Covidence #,Study,Notes,Tags
0,Nonvocational Outcomes from a Randomized Contr...,"Ferguson, Kristin M.",Purpose: This randomized controlled trial comp...,2018.0,NaN,Research on Social Work Practice,28,5,603-618,"SAGE Publications. 2455 Teller Road, Thousand ...",10.1177/1049731517709076,NaN,#5,Ferguson 2018,NaN,NaN
1,Increasing Identification of Homeless Students...,"Shephard, Daniel D.; Hall, Crystal C.; Lambert...",Over 1.5 million students in the United States...,2021.0,NaN,Educational Researcher,50,4,239-248,"SAGE Publications. 2455 Teller Road, Thousand ...",10.3102/0013189X20981067,NaN,#21,Shephard 2021,NaN,NaN
2,'I Need to Get My Culture Back': Youth and Pro...,Charlene Kuo; Michelle Jasczynski; Jee Hun Yoo...,There is growing interest in decolonizing sexu...,2023.0,NaN,Prevention Science,24,2,209-221,Springer. Available from: Springer Nature. One...,10.1007/s11121-023-01573-7,NaN,#40,CharleneKuo 2023,NaN,NaN
3,Extended foster care and homelessness: Assessi...,"Spindle-Jackson, A.; Byrne, T.; Collins, M.E.",Youth aging out of foster care experience high...,2024.0,NaN,Child. Youth Serv. Rev.,164,NaN,NaN,NaN,10.1016/j.childyouth.2024.107820,NaN,#77,Spindle-Jackson 2024,NaN,NaN
4,A feasibility (pilot) mixed methods study of a...,"Vasudev, A.; Ionson, E.; Sathiaselan, J.; That...",Background: Various service provision models f...,2024.0,NaN,Pilot Feasibility Stud.,10,1,NaN,NaN,10.1186/s40814-024-01452-0,NaN,#79,Vasudev 2024,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1837,An eight-year epidemiologic study of head and ...,"Qian, Xu; Nguyen, Duc T.; Albers, Andreas E.; ...",BACKGROUND: Head and neck tuberculosis (HNTB) ...,2019.0,NaN,Tuberculosis (Edinb),116S,NaN,S71-S77,NaN,10.1016/j.tube.2019.04.013,NaN,#3968,Qian 2019,NaN,NaN
1838,Assessing Racial Disparities in HCV Infection ...,"McGonigle, Keanan; Carley, Tess; Hoff, Clarissa",OBJECTIVES: This study assessed racial dispari...,2018.0,NaN,J Racial Ethn Health Disparities,5,5,1052-1058,NaN,10.1007/s40615-017-0453-y,NaN,#3969,McGonigle 2018,NaN,NaN
1839,Healthcare resource utilization and costs asso...,"Samant, Salome; Chen, Edith; Carias, Cristina;...",AIM: To investigate hepatitis A-related health...,2024.0,NaN,J Med Econ,27,1,1046-1052,NaN,10.1080/13696998.2024.2384263,NaN,#3970,Samant 2024,NaN,NaN
1840,Technology Access and Perceptions of Telehealt...,"Ertl, Melissa M.; Jones, Alexis; Hickson, Robe...",PURPOSE: This study examined access to technol...,2024.0,NaN,J Adolesc Health,74,3,582-590,NaN,10.1016/j.jadohealth.2023.09.019,NaN,#3971,Ertl 2024,NaN,NaN


In [36]:
#Prompt: Zero shots-direct decision-binary
#Test different models (below)

start = time.perf_counter()
print(start)

# For phase 1 test, add these two columns in study_details below
#         f"Author Affiliation: {row['author_affiliation']}\n"
#         f"Keywords: {row['keywords']}"

def get_decision_for_row(row, row_index):
    study_details = (
        f"Title: {row['title']}\n"
        f"Abstract: {row['abstract']}\n"
        f"Publication Year: {row['publication_year']}\n"
        f"Journal Name: {row['journal_name']}\n"
        f"Author: {row['author']}\n"

    )

    prompt = f"""
You are an expert reviewer for research studies with human-like reasoning. Your goal is to assist in pre-screening abstracts to identify studies that might meet inclusion criteria for a systematic review of youth homelessness interventions. This is an initial triage, not a final eligibility decision.

When abstracts are ambiguous or incomplete, lean toward inclusion for further review to minimize false negatives.

---

Study Details:
{study_details}

---

Pre-Screening Focus:
1. Original/Empirical Research — Study is based on primary data or direct observations. Exclude editorials, letters, protocols, systematic reviews, meta-analyses, and studies that rely only on secondary data or reanalysis of existing datasets.

2. Target Population (Age) — Youth aged 13–25. Include if:
- The study uses terms like "youth," "adolescents," "young people," or related terms.
- The study includes a mixed-age sample that may cover youth (e.g., 16–30, 18–35).
- No age is mentioned but the population is described as youth-related.
- The intervention setting or system serves youth or at-risk young people (e.g., schools, shelters, child welfare systems).

Exclude only if the study explicitly states it focuses solely on ages clearly outside 13–25 (e.g., only participants over 30).

3. Target Population (Homelessness) — Include if:
- The population is homeless, unstably housed, or at risk (include related terms like "runaway," "refugee minor," "vulnerably housed," "shelter youth").
- The population is served by systems addressing homelessness (e.g., McKinney-Vento programs, Housing First, shelters, foster care transitions).
- The study addresses social, educational, or health programs aimed at populations known to be at high risk of homelessness.
- The population is described as disadvantaged or vulnerable and housing status is unclear, lean toward inclusion.

Exclude only if the population is explicitly stated to have no connection to homelessness or housing risk.

4. Program/Service/Intervention Focus — Include interventions addressing any issue (e.g., education, mental health, substance use, health access) when delivered to, or within systems that serve, homeless or at-risk youth. Do not limit inclusion to programs that target homelessness outcomes directly.

Other criteria (effectiveness reporting, implementation reporting, full geographical confirmation) will be checked at full-text review. Do not exclude based on these at this stage.

---

Decision Instructions:
- Recommend Include when study details match or might match the pre-screening focus areas.
- Recommend Exclude only if there is clear evidence that the study does not match focus areas.
- When in doubt, lean toward Include to minimize false negatives.

Return a list of codes indicating which criteria were not met (for Exclude):
- X-1: Not original/empirical research (e.g., editorials, letters, protocols, reviews, meta-analyses, secondary data studies)
- X-2: Target population explicitly not youth (e.g., study only includes ages outside 13–25)
- X-3: Population explicitly not homeless/at risk
- X-4: No program/service/intervention focus

---

Return a JSON object in the following format:
{{
  "Decision": "Include" | "Exclude",
  "Explanation_Codes": ["X-code", ...] or [],
  "Focus_Area_Results": {{
    "Original_Empirical": "Yes" | "No",
    "Youth_Age_13_25": "Yes" | "No",
    "Homelessness_Target": "Yes" | "No",
    "Program_Focus": "Yes" | "No"
  }}
}}


Return only the JSON object, no extra text.
    """

    try:
        response = client.chat.completions.create(
            #try different models here
            #"gpt-5-mini-2025-08-07"
            #"gpt-5-nano-2025-08-07"
            #"gpt-5.2-2025-12-11"
            model="gpt-5-nano-2025-08-07",
            messages=[
                {"role": "system", "content": "You are an expert reviewer with human-like reasoning."},
                {"role": "user", "content": prompt}
            ],
            
        )
        reply = response.choices[0].message.content.strip()

        if reply.startswith("```json"):
            reply = reply.replace("```json", "").strip("` \n")
        elif reply.startswith("```"):
            reply = reply.strip("` \n")

        result = json.loads(reply)
        return (
            result.get("Decision", "Exclude"),
            json.dumps(result.get("Explanation_Codes", [])),
            json.dumps(result.get("Focus_Area_Results", {}))
        )

    except json.JSONDecodeError:
        print(f"JSON parse error on row {row_index + 1}")
        print("GPT Reply:\n", reply)
        return "Exclude", json.dumps(["Parse error"]), json.dumps({})
    except Exception as e:
        print(f"Error on row {row_index + 1}: {e}")
        return "Exclude", json.dumps(["API error"]), json.dumps({})

# Process the data
decisions = []
explanation_codes = []
focus_area_logs = []

for i, row in df.iterrows():
    decision, explanation, focus_areas = get_decision_for_row(row, i)
    decisions.append(decision)
    explanation_codes.append(explanation)
    focus_area_logs.append(focus_areas)
   

df["Decision"] = decisions
df["Explanation_Codes"] = explanation_codes
df["Focus_Area_Results"] = focus_area_logs

output_path = "Phase2_zero_direct_binary_GPT_5_nano.xlsx" # change this name as appropriate
df.to_excel(output_path, index=False)
print(f"\n Processing complete. File saved to: {output_path}")

# calculate elapsed time
print(start)
end = time.perf_counter()
elapsed = end - start
print(f"Code took {elapsed:.4f} seconds")


3141068.985708166
Error on row 694: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}

 Processing complete. File saved to: Phase2_zero_direct_binary_GPT_5_nano.xlsx
3141068.985708166
Code took 16395.1143 seconds


In [37]:
import pandas as pd

# Load data
AI = pd.read_excel("Phase2_zero_direct_binary_GPT_5_nano.xlsx") #change the data name here as appropriate 
human = pd.read_csv("Phase_2_combined.csv")
human.rename(columns={"Title": "title", "Abstract": "abstract", "Published Year": "publication_year", "Journal": "journal_name", "Authors": "author"}, inplace=True)

# Merge by index (or adjust if you have an ID column)
combined = pd.concat([AI, human], axis=1)

# Drop duplicate columns (if any columns are fully redundant / duplicated across datasets)
# Create a set of columns to keep: drop any duplicates based on column name and identical content
# We'll detect duplicated columns by value
def drop_redundant_columns(df):
    cols_to_drop = []
    for i, col1 in enumerate(df.columns):
        for col2 in df.columns[i + 1:]:
            if col1 != col2 and df[col1].equals(df[col2]):
                cols_to_drop.append(col2)
    return df.drop(columns=set(cols_to_drop))

combined = drop_redundant_columns(combined)

# Create alignment column
def classify_alignment(row):
    ai_decision = row["Decision"]
    human_included = row["is.included"]
    
    if ai_decision == "Include" and human_included == 1:
        return "Aligned: Both Include"
    elif ai_decision == "Exclude" and human_included == 0:
        return "Aligned: Both Exclude"
    elif ai_decision == "Include" and human_included == 0:
        return "AI Include, Human Exclude"
    elif ai_decision == "Exclude" and human_included == 1:
        return "AI Exclude, Human Include"
    else:
        return "Other"

combined["Alignment"] = combined.apply(classify_alignment, axis=1)

# Show counts
alignment_counts = combined["Alignment"].value_counts()
print(alignment_counts)

# Save to file
combined.to_excel("Phase2_alignment_check_zero_direct_binary_GPT_5_nano.xlsx", index=False) # change this name as appropriate
print("\n Alignment file saved as Phase2_alignment_check_zero_direct_binary_GPT_5_nano.xlsx")


Alignment
AI Include, Human Exclude    984
Aligned: Both Exclude        626
Aligned: Both Include        217
AI Exclude, Human Include     15
Name: count, dtype: int64

 Alignment file saved as Phase2_alignment_check_zero_direct_binary_GPT_5_nano.xlsx


In [38]:
# Calculate confusion matrix components
TP = combined[(combined["Decision"] == "Include") & (combined["is.included"] == 1)].shape[0]
FP = combined[(combined["Decision"] == "Include") & (combined["is.included"] == 0)].shape[0]
TN = combined[(combined["Decision"] == "Exclude") & (combined["is.included"] == 0)].shape[0]
FN = combined[(combined["Decision"] == "Exclude") & (combined["is.included"] == 1)].shape[0]

# Calculate metrics
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0  # Sensitivity
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
accuracy = (TP + TN) / (TP + FP + TN + FN) if (TP + FP + TN + FN) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Print results
print("\nConfusion Matrix Counts:")
print(f"True Positives (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print(f"False Negatives (FN): {FN}")

print("\nAI Effectiveness Metrics for Systematic Review Pre-screening:")
print(f"Precision (Positive Predictive Value): {precision:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Accuracy: {accuracy:.3f}")
print(f"F1 Score: {f1:.3f}")



Confusion Matrix Counts:
True Positives (TP): 217
False Positives (FP): 984
True Negatives (TN): 626
False Negatives (FN): 15

AI Effectiveness Metrics for Systematic Review Pre-screening:
Precision (Positive Predictive Value): 0.181
Recall (Sensitivity): 0.935
Specificity: 0.389
Accuracy: 0.458
F1 Score: 0.303


In [39]:

#zero shots-direct decision-Likert 

start = time.perf_counter()
print(start)

# for phase 1 test, add these two columns in study_details below
#         f"Author Affiliation: {row['author_affiliation']}\n"
#         f"Keywords: {row['keywords']}"

def get_decision_for_row(row, row_index):
    study_details = (
        f"Title: {row['title']}\n"
        f"Abstract: {row['abstract']}\n"
        f"Publication Year: {row['publication_year']}\n"
        f"Journal Name: {row['journal_name']}\n"
        f"Author: {row['author']}\n"

    )

    prompt = f"""
You are an expert reviewer assisting with abstract pre-screening for a systematic review of youth homelessness interventions. This is an initial triage stage, not a final eligibility decision.

Your goal is to estimate how likely this study should be INCLUDED for full-text review.

When abstracts are ambiguous or incomplete, lean toward higher likelihood scores to minimize false negatives.

---

Study Details:
{study_details}

---

Pre-Screening Focus:
1. Original/Empirical Research — Based on primary data or direct observations. Exclude editorials, letters, protocols, systematic reviews, meta-analyses, or studies relying only on secondary data.

2. Target Population (Age) — Youth aged 13–25. Include if youth-related terms are used, mixed-age samples may include youth, or the setting/system serves youth. Exclude only if the study explicitly focuses solely on ages clearly outside 13–25.

3. Target Population (Homelessness) — Include if the population is homeless, unstably housed, at risk of homelessness, or served by systems addressing homelessness risk. If housing status is unclear but the population is described as vulnerable or disadvantaged, lean toward inclusion.

4. Program/Service/Intervention Focus — Include any program, service, or intervention delivered within systems serving homeless or at-risk youth, regardless of whether housing outcomes are directly targeted.

Other criteria (effectiveness, implementation, geography) will be assessed at full-text review.

---

Scoring Instructions:
Rate the likelihood that this study should be INCLUDED for full-text review using the following scale:

1 = Very unlikely to be relevant  
2 = Unlikely to be relevant  
3 = Possibly relevant  
4 = Likely relevant  
5 = Very likely relevant  

Use the full range of the scale.
When in doubt, assign a higher score to minimize false negatives.

---

Return a JSON object in the following format:
{{
  "Relevance_Score": 1,
  "Focus_Area_Results": {{
    "Original_Empirical": "Yes" | "No",
    "Youth_Age_13_25": "Yes" | "No",
    "Homelessness_Target": "Yes" | "No",
    "Program_Focus": "Yes" | "No"
  }}
}}

Return only the JSON object. No extra text.
    """

    try:
        response = client.chat.completions.create(
            #try different models here
            #"gpt-5-mini-2025-08-07"
            #"gpt-5-nano-2025-08-07"
            #"gpt-5.2-2025-12-11"
            model="gpt-5-nano-2025-08-07",
            messages=[
                {"role": "system", "content": "You are an expert reviewer with human-like reasoning."},
                {"role": "user", "content": prompt}
            ],
        )

        reply = response.choices[0].message.content.strip()

        # Clean markdown if present
        if reply.startswith("```json"):
            reply = reply.replace("```json", "").strip("` \n")
        elif reply.startswith("```"):
            reply = reply.strip("` \n")

        result = json.loads(reply)

        return (
            result.get("Relevance_Score", None),
            json.dumps(result.get("Focus_Area_Results", {}))
        )

    except json.JSONDecodeError:
        print(f"JSON parse error on row {row_index + 1}")
        print("GPT Reply:\n", reply)
        return None, json.dumps({})
    except Exception as e:
        print(f"Error on row {row_index + 1}: {e}")
        return None, json.dumps({})


# Run across dataset
scores = []
focus_area_logs = []

for i, row in df.iterrows():
    score, focus_areas = get_decision_for_row(row, i)
    scores.append(score)
    focus_area_logs.append(focus_areas)
  

df["Relevance_Score"] = scores
df["Focus_Area_Results"] = focus_area_logs

output_path = "Phase2_zero_direct_likert_GPT_5_nano.xlsx"
df.to_excel(output_path, index=False)
print(f"\n Processing complete. File saved to: {output_path}")

# calculate elapsed time
print(start)
end = time.perf_counter()
elapsed = end - start
print(f"Code took {elapsed:.4f} seconds")


3180927.010978375

 Processing complete. File saved to: Phase2_zero_direct_likert_GPT_5_nano.xlsx
3180927.010978375
Code took 14733.3477 seconds


In [40]:
AI = pd.read_excel("Phase2_zero_direct_likert_GPT_5_nano.xlsx")
human = pd.read_csv("Phase_2_combined.csv")
human.rename(columns={"Title": "title", "Abstract": "abstract", "Published Year": "publication_year", "Journal": "journal_name", "Authors": "author"}, inplace=True)

# Define Likert → binary threshold
LIKERT_INCLUDE_THRESHOLD = 3 

AI["AI_Binary_Decision"] = AI["Relevance_Score"].apply(
    lambda x: "Include" if x >= LIKERT_INCLUDE_THRESHOLD else "Exclude"

)

combined = pd.concat([AI, human], axis=1)
combined = drop_redundant_columns(combined)
def classify_alignment(row):
    ai_decision = row["AI_Binary_Decision"]
    human_included = row["is.included"]

    if ai_decision == "Include" and human_included == 1:
        return "Aligned: Both Include"
    elif ai_decision == "Exclude" and human_included == 0:
        return "Aligned: Both Exclude"
    elif ai_decision == "Include" and human_included == 0:
        return "AI Include, Human Exclude"
    elif ai_decision == "Exclude" and human_included == 1:
        return "AI Exclude, Human Include"
    else:
        return "Other"
combined["Alignment"] = combined.apply(classify_alignment, axis=1)
alignment_counts = combined["Alignment"].value_counts()
print(alignment_counts)

combined.to_excel(
    f"Phase2_alignment_check_likert_threshold_{LIKERT_INCLUDE_THRESHOLD}_GPT_5_nano.xlsx", # change this name as appropriate
    index=False
)

print("Alignment file saved")



Alignment
AI Include, Human Exclude    1192
Aligned: Both Exclude         418
Aligned: Both Include         214
AI Exclude, Human Include      18
Name: count, dtype: int64
Alignment file saved


In [41]:
# Calculate confusion matrix components using Likert-derived binary decision

TP = combined[
    (combined["AI_Binary_Decision"] == "Include") &
    (combined["is.included"] == 1)
].shape[0]

FP = combined[
    (combined["AI_Binary_Decision"] == "Include") &
    (combined["is.included"] == 0)
].shape[0]

TN = combined[
    (combined["AI_Binary_Decision"] == "Exclude") &
    (combined["is.included"] == 0)
].shape[0]

FN = combined[
    (combined["AI_Binary_Decision"] == "Exclude") &
    (combined["is.included"] == 1)
].shape[0]


# Calculate metrics
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0  # Sensitivity
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
accuracy = (TP + TN) / (TP + FP + TN + FN) if (TP + FP + TN + FN) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0


# Print results
print(f"\nLikert Threshold ≥ {LIKERT_INCLUDE_THRESHOLD}")
print("Confusion Matrix Counts:")
print(f"True Positives (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print(f"False Negatives (FN): {FN}")

print("\nAI Effectiveness Metrics for Abstract Pre-screening:")
print(f"Precision (Positive Predictive Value): {precision:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Accuracy: {accuracy:.3f}")
print(f"F1 Score: {f1:.3f}")



Likert Threshold ≥ 3
Confusion Matrix Counts:
True Positives (TP): 214
False Positives (FP): 1192
True Negatives (TN): 418
False Negatives (FN): 18

AI Effectiveness Metrics for Abstract Pre-screening:
Precision (Positive Predictive Value): 0.152
Recall (Sensitivity): 0.922
Specificity: 0.260
Accuracy: 0.343
F1 Score: 0.261


In [42]:
#Zero-shot-COT-binary

start = time.perf_counter()
print(start)

# For phase 1 test, add these two columns in study_details below
#         f"Author Affiliation: {row['author_affiliation']}\n"
#         f"Keywords: {row['keywords']}"

def get_decision_for_row(row, row_index):
    study_details = (
        f"Title: {row['title']}\n"
        f"Abstract: {row['abstract']}\n"
        f"Publication Year: {row['publication_year']}\n"
        f"Journal Name: {row['journal_name']}\n"
        f"Author: {row['author']}\n"

    )

    # NOTE: double braces {{ }} are REQUIRED inside f-strings
    prompt = f"""
You are an expert reviewer conducting abstract pre-screening for a systematic review of youth homelessness interventions.

This is an INITIAL TRIAGE stage, not a final eligibility decision.

You MUST internally evaluate the study step by step using the screening process below before making a decision.
Do NOT output your reasoning.

When information is ambiguous or incomplete at any step, LEAN TOWARD INCLUSION to minimize false negatives.

---

Study Details:
{study_details}

---

Step-by-Step Screening Process:

Step 1: Original / Empirical Research  
Determine whether the study reports original empirical research based on primary data or direct observation.  
Exclude editorials, letters, protocols, systematic reviews, meta-analyses, or studies relying only on secondary data.

Step 2: Target Population (Age)  
Determine whether the study focuses on youth aged 13–25.  
Include if youth-related terms are used, if the sample includes mixed ages that may include youth, or if the intervention setting serves youth.  
Exclude ONLY if the study explicitly focuses solely on ages clearly outside 13–25.

Step 3: Target Population (Homelessness)  
Determine whether the population is homeless, unstably housed, at risk of homelessness, or served by systems addressing homelessness risk.  
If housing status is unclear but the population is described as vulnerable or disadvantaged, lean toward inclusion.  
Exclude ONLY if the population is explicitly stated to have no connection to homelessness or housing risk.

Step 4: Program / Service / Intervention  
Determine whether the study describes a program, service, or intervention delivered to or within systems serving homeless or at-risk youth.  
Do NOT require the intervention to directly target homelessness outcomes.

---

Final Decision Rule:
- If ANY step is clearly failed → Exclude  
- If ALL steps are met or ambiguous → Include  

---

Return ONLY the final decision in the following JSON format.
DO NOT include explanations or reasoning.

{{
  "Decision": "Include" | "Exclude",
  "Explanation_Codes": ["X-code", ...] or [],
  "Focus_Area_Results": {{
    "Original_Empirical": "Yes" | "No",
    "Youth_Age_13_25": "Yes" | "No",
    "Homelessness_Target": "Yes" | "No",
    "Program_Focus": "Yes" | "No"
  }}
}}
"""

    try:
        response = client.chat.completions.create(
            #try different models here
            #"gpt-5-mini-2025-08-07"
            #"gpt-5-nano-2025-08-07"
            #"gpt-5.2-2025-12-11"
            model="gpt-5-nano-2025-08-07",
            messages=[
                {"role": "system", "content": "You are an expert reviewer with rigorous, structured reasoning."},
                {"role": "user", "content": prompt}
            ],
        )

        reply = response.choices[0].message.content.strip()

        # Remove markdown fences if present
        if reply.startswith("```json"):
            reply = reply.replace("```json", "").strip("` \n")
        elif reply.startswith("```"):
            reply = reply.strip("` \n")

        result = json.loads(reply)

        return (
            result.get("Decision", "Exclude"),
            json.dumps(result.get("Explanation_Codes", [])),
            json.dumps(result.get("Focus_Area_Results", {}))
        )

    except json.JSONDecodeError:
        print(f"⚠️ JSON parse error on row {row_index + 1}")
        print("GPT Reply:\n", reply)
        return "Exclude", json.dumps(["Parse error"]), json.dumps({})
    except Exception as e:
        print(f"❌ Error on row {row_index + 1}: {e}")
        return "Exclude", json.dumps(["API error"]), json.dumps({})


# ==============================
# RUN PIPELINE
# ==============================

decisions = []
explanation_codes = []
focus_area_logs = []

for i, row in df.iterrows():
    decision, explanation, focus_areas = get_decision_for_row(row, i)
    decisions.append(decision)
    explanation_codes.append(explanation)
    focus_area_logs.append(focus_areas)
    time.sleep(1)  # optional rate limit safety

df["Decision"] = decisions
df["Explanation_Codes"] = explanation_codes
df["Focus_Area_Results"] = focus_area_logs

output_path = "Phase2_zero_shot_CoT_binary_GPT_5_nano.xlsx"
df.to_excel(output_path, index=False)

print(f"\n Processing complete. File saved to: {output_path}")

# calculate elapsed time
print(start)
end = time.perf_counter()
elapsed = end - start
print(f"Code took {elapsed:.4f} seconds")

3199076.360699375

 Processing complete. File saved to: Phase2_zero_shot_CoT_binary_GPT_5_nano.xlsx
3199076.360699375
Code took 17606.1263 seconds


In [43]:
import pandas as pd

# Load data
AI = pd.read_excel("Phase2_zero_shot_CoT_binary_GPT_5_nano.xlsx") # change name as appropriate
human = pd.read_csv("Phase_2_combined.csv")
human.rename(columns={"Title": "title", "Abstract": "abstract", "Published Year": "publication_year", "Journal": "journal_name", "Authors": "author"}, inplace=True)

# Merge by index (or ID if you later add one)
combined = pd.concat([AI, human], axis=1)

# Drop duplicated columns (same content)
def drop_redundant_columns(df):
    cols_to_drop = []
    for i, col1 in enumerate(df.columns):
        for col2 in df.columns[i + 1:]:
            if col1 != col2 and df[col1].equals(df[col2]):
                cols_to_drop.append(col2)
    return df.drop(columns=set(cols_to_drop))

combined = drop_redundant_columns(combined)

# Create alignment column
def classify_alignment(row):
    ai_decision = row["Decision"]
    human_included = row["is.included"]

    if ai_decision == "Include" and human_included == 1:
        return "Aligned: Both Include"      # True Positive
    elif ai_decision == "Exclude" and human_included == 0:
        return "Aligned: Both Exclude"      # True Negative
    elif ai_decision == "Include" and human_included == 0:
        return "AI Include, Human Exclude"  # False Positive
    elif ai_decision == "Exclude" and human_included == 1:
        return "AI Exclude, Human Include"  # False Negative
    else:
        return "Other"

combined["Alignment"] = combined.apply(classify_alignment, axis=1)

# Show counts
alignment_counts = combined["Alignment"].value_counts()
print(alignment_counts)

# Save output
combined.to_excel(
    "Phase2_alignment_check_ZS_CoT_binary_GPT_5_nano.xlsx", # change this name as appropriate
    index=False
)

Alignment
Aligned: Both Exclude        1206
AI Include, Human Exclude     404
Aligned: Both Include         194
AI Exclude, Human Include      38
Name: count, dtype: int64


In [44]:
# Confusion matrix components (ZS-CoT-Binary)

TP = combined[
    (combined["Decision"] == "Include") &
    (combined["is.included"] == 1)
].shape[0]

FP = combined[
    (combined["Decision"] == "Include") &
    (combined["is.included"] == 0)
].shape[0]

TN = combined[
    (combined["Decision"] == "Exclude") &
    (combined["is.included"] == 0)
].shape[0]

FN = combined[
    (combined["Decision"] == "Exclude") &
    (combined["is.included"] == 1)
].shape[0]


# Metrics
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0  # Sensitivity
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
accuracy = (TP + TN) / (TP + FP + TN + FN) if (TP + FP + TN + FN) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0


# Print results
print("\nZS-CoT-Binary (GPT-5-mini) — Confusion Matrix:")
print(f"True Positives (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print(f"False Negatives (FN): {FN}")

print("\nAI Effectiveness Metrics for Abstract Pre-screening:")
print(f"Precision (PPV): {precision:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Accuracy: {accuracy:.3f}")
print(f"F1 Score: {f1:.3f}")



ZS-CoT-Binary (GPT-5-mini) — Confusion Matrix:
True Positives (TP): 194
False Positives (FP): 404
True Negatives (TN): 1206
False Negatives (FN): 38

AI Effectiveness Metrics for Abstract Pre-screening:
Precision (PPV): 0.324
Recall (Sensitivity): 0.836
Specificity: 0.749
Accuracy: 0.760
F1 Score: 0.467


In [45]:
#Zero-shot-COT-Likert 

start = time.perf_counter()
print(start)

# For phase 1 test, add these two columns in study_details below
#         f"Author Affiliation: {row['author_affiliation']}\n"
#         f"Keywords: {row['keywords']}"

def get_decision_for_row(row, row_index):

    study_details = (
        f"Title: {row['title']}\n"
        f"Abstract: {row['abstract']}\n"
        f"Publication Year: {row['publication_year']}\n"
        f"Journal Name: {row['journal_name']}\n"
        f"Author: {row['author']}\n"
    )

    prompt = f"""
You are an expert reviewer assisting with abstract pre-screening for a systematic review of youth homelessness interventions.

This is an INITIAL TRIAGE stage, not a final eligibility decision.

You MUST internally reason step by step using the screening process below before assigning a score.
Do NOT output your reasoning.

When information is ambiguous or incomplete at any step, LEAN TOWARD HIGHER SCORES to minimize false negatives.

---

Study Details:
{study_details}

---

Step-by-Step Screening Process:

Step 1: Original / Empirical Research  
Determine whether the study reports original empirical research based on primary data or direct observation.  
Exclude editorials, letters, protocols, systematic reviews, meta-analyses, or studies relying only on secondary data.

Step 2: Target Population (Age)  
Determine whether the study focuses on youth aged 13–25.  
Include if youth-related terms are used, if the sample includes mixed ages that may include youth, or if the intervention setting serves youth.  
Exclude ONLY if the study explicitly focuses solely on ages clearly outside 13–25.

Step 3: Target Population (Homelessness)  
Determine whether the population is homeless, unstably housed, at risk of homelessness, or served by systems addressing homelessness risk.  
If housing status is unclear but the population is described as vulnerable or disadvantaged, lean toward inclusion.  
Exclude ONLY if the population is explicitly stated to have no connection to homelessness or housing risk.

Step 4: Program / Service / Intervention  
Determine whether the study describes a program, service, or intervention delivered to or within systems serving homeless or at-risk youth.  
Do NOT require the intervention to directly target homelessness outcomes.

---

Scoring Instructions:

After completing all steps above, assign a likelihood score indicating how appropriate this study is for FULL-TEXT REVIEW.

Use the following scale:

1 = Very unlikely to be relevant  
2 = Unlikely to be relevant  
3 = Possibly relevant  
4 = Likely relevant  
5 = Very likely relevant  

Guidance:
- If ALL steps are clearly met → 4 or 5  
- If steps are mixed or ambiguous → 3 or 4  
- If ONE step is weak but not definitively failed → at least 3  
- Assign 1 or 2 ONLY if relevance is clearly unlikely

---

Return ONLY the final result in the following JSON format.
DO NOT include explanations or reasoning.

{{
  "Relevance_Score": 1,
  "Focus_Area_Results": {{
    "Original_Empirical": "Yes" | "No",
    "Youth_Age_13_25": "Yes" | "No",
    "Homelessness_Target": "Yes" | "No",
    "Program_Focus": "Yes" | "No"
  }}
}}
"""

    try:
        response = client.chat.completions.create(
            #try different models here
            #"gpt-5-mini-2025-08-07"
            #"gpt-5-nano-2025-08-07"
            #"gpt-5.2-2025-12-11"
            model="gpt-5-nano-2025-08-07",
            messages=[
                {"role": "system", "content": "You are an expert reviewer with rigorous, structured reasoning."},
                {"role": "user", "content": prompt}
            ],
        )

        reply = response.choices[0].message.content.strip()

        # Clean markdown fences if present
        if reply.startswith("```json"):
            reply = reply.replace("```json", "").strip("` \n")
        elif reply.startswith("```"):
            reply = reply.strip("` \n")

        result = json.loads(reply)

        return (
            result.get("Relevance_Score", None),
            json.dumps(result.get("Focus_Area_Results", {}))
        )

    except json.JSONDecodeError:
        print(f"⚠️ JSON parse error on row {row_index + 1}")
        print("GPT Reply:\n", reply)
        return None, json.dumps({})
    except Exception as e:
        print(f"❌ Error on row {row_index + 1}: {e}")
        return None, json.dumps({})


# ==============================
# RUN PIPELINE
# ==============================

scores = []
focus_area_logs = []

for i, row in df.iterrows():
    score, focus_areas = get_decision_for_row(row, i)
    scores.append(score)
    focus_area_logs.append(focus_areas)
    time.sleep(1)  # optional rate-limit safety

df["Relevance_Score"] = scores
df["Focus_Area_Results"] = focus_area_logs

output_path = "Phase2_zero_shot_CoT_likert_GPT_5_nano.xlsx"
df.to_excel(output_path, index=False)

print(f"\n Processing complete. File saved to: {output_path}")

# calculate elapsed time
print(start)
end = time.perf_counter()
elapsed = end - start
print(f"Code took {elapsed:.4f} seconds")

3230345.506697
⚠️ JSON parse error on row 14
GPT Reply:
 {
  "Relevance_Score": 3,
  "Focus_Area_Results": {
    "Original_Empirical": "Yes",
    "Youth_Age_13_25": "Yes",
    "Homelessness_Target": "Yes",
n    "Program_Focus": "No"
  }
}

 Processing complete. File saved to: Phase2_zero_shot_CoT_likert_GPT_5_nano.xlsx
3230345.506697
Code took 19507.7892 seconds


In [46]:
import pandas as pd

# Load CoT-Likert AI output
AI = pd.read_excel("Phase2_zero_shot_CoT_likert_GPT_5_nano.xlsx")
human = pd.read_csv("Phase_2_combined.csv")
human.rename(columns={"Title": "title", "Abstract": "abstract", "Published Year": "publication_year", "Journal": "journal_name", "Authors": "author"}, inplace=True)


LIKERT_INCLUDE_THRESHOLD = 3

# Create AI binary decision
AI["AI_Binary_Decision"] = AI["Relevance_Score"].apply(
    lambda x: "Include" if x >= LIKERT_INCLUDE_THRESHOLD else "Exclude"
)

# Merge FIRST
combined = pd.concat([AI, human], axis=1)

# Alignment (BEFORE dropping columns)
def classify_alignment(row):
    ai_decision = row["AI_Binary_Decision"]
    human_included = row["is.included"]

    if ai_decision == "Include" and human_included == 1:
        return "Aligned: Both Include"
    elif ai_decision == "Exclude" and human_included == 0:
        return "Aligned: Both Exclude"
    elif ai_decision == "Include" and human_included == 0:
        return "AI Include, Human Exclude"
    elif ai_decision == "Exclude" and human_included == 1:
        return "AI Exclude, Human Include"
    else:
        return "Other"

combined["Alignment"] = combined.apply(classify_alignment, axis=1)

print(combined["Alignment"].value_counts())

# NOW drop redundant columns (optional, cosmetic)
def drop_redundant_columns(df):
    cols_to_drop = []
    for i, col1 in enumerate(df.columns):
        for col2 in df.columns[i + 1:]:
            if col1 != col2 and df[col1].equals(df[col2]):
                cols_to_drop.append(col2)
    return df.drop(columns=set(cols_to_drop))

combined = drop_redundant_columns(combined)

combined.to_excel(
    f"Phase2_alignment_check_ZS_CoT_likert_threshold_{LIKERT_INCLUDE_THRESHOLD}_GPT_5_nano.xlsx", # change this name as appropriate
    index=False
)

print("Alignment file saved")


Alignment
AI Include, Human Exclude    1335
Aligned: Both Exclude         275
Aligned: Both Include         223
AI Exclude, Human Include       9
Name: count, dtype: int64
Alignment file saved


In [47]:
# --- SAFETY CHECK ---
if "AI_Binary_Decision" not in combined.columns:
    combined["AI_Binary_Decision"] = combined["Relevance_Score"].apply(
        lambda x: "Include" if x >= LIKERT_INCLUDE_THRESHOLD else "Exclude"
    )

# --- Confusion matrix ---
TP = combined[
    (combined["AI_Binary_Decision"] == "Include") &
    (combined["is.included"] == 1)
].shape[0]

FP = combined[
    (combined["AI_Binary_Decision"] == "Include") &
    (combined["is.included"] == 0)
].shape[0]

TN = combined[
    (combined["AI_Binary_Decision"] == "Exclude") &
    (combined["is.included"] == 0)
].shape[0]

FN = combined[
    (combined["AI_Binary_Decision"] == "Exclude") &
    (combined["is.included"] == 1)
].shape[0]

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
accuracy = (TP + TN) / (TP + FP + TN + FN) if (TP + FP + TN + FN) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nZS-CoT-Likert (threshold ≥ {LIKERT_INCLUDE_THRESHOLD})")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Accuracy: {accuracy:.3f}")
print(f"F1: {f1:.3f}")



ZS-CoT-Likert (threshold ≥ 3)
TP: 223, FP: 1335, TN: 275, FN: 9
Precision: 0.143
Recall: 0.961
Specificity: 0.171
Accuracy: 0.270
F1: 0.249
